# SAM RCE runs: isentropic streamfunction in $\theta_e$ and frozen $\theta_e$

Each experiment is one SAM run directory: the 3D snapshots are the files `OUT_3D/<prefix>*.nc` (one or more
times per file, concatenated in time), and the density profile comes from the run's reference profile
(`RHO` in the `OUT_STAT` file, at the time closest to each snapshot).

The streamfunction is computed with two vertical coordinates:

- $\theta_e$: `thermo.equivalent_potential_temperature` (liquid water reference, Emanuel 1994 with ice),
- $\theta_{ef}$: `thermo.frozen_equivalent_potential_temperature` (ice reference, Pauluis 2018).

Both are computed from the same snapshot, so each file is read only once. The conditional sums
keep the experiment and time dependence, as in `isentropic_streamfunction_frozen.ipynb`.

Use the **Python (iso)** kernel.

## Configuration

In [ ]:
from pathlib import Path

ROOT = Path("/scratch/op13/SAM-Torch")

# experiment ID -> (run directory, file prefix)
RUNS = {
    "RCE_REF": (ROOT / "RCE_REF", "RCE_REF_64"),
    "RCE_REF_TKE_M2005": (ROOT / "RCE_REF_TKE_M2005", "RCE_REF_TKE_M2005_300K_64"),
}
FILES_3D = "OUT_3D/{prefix}*.nc"
FILE_STAT = "OUT_STAT/{prefix}*.nc"   # reference profile with RHO(time, z); set to None to compute the density
RHO_VAR = "RHO"

# SAM 6.11 3D snapshot output
#   W (m/s), TABS (K), QV, QN (g/kg), p(z) (mb), PP (Pa)
VAR = dict(w="W", T="TABS", p="p", pp="PP", qv="QV", qn="QN")
Z_DIM = "z"
TIME_DIM = "time"
T_ICE, T_LIQ = 253.16, 273.16   # SAM's partition of QN into ice and liquid

# Snapshots to process in each run (counted over all its files), e.g. slice(-48, None, 4) for a quick test
TIME_SLICE = slice(None)

# Bins (K) for each coordinate. The frozen theta_e is higher than theta_e for moist air
# (vapor carries the latent heat of sublimation), so check the ranges printed below and adjust.
BINS = {
    "theta_e":  dict(th_min=300.0, th_max=370.0, nbin=141),
    "theta_ef": dict(th_min=300.0, th_max=390.0, nbin=181),
}
LABEL = {"theta_e": r"$\theta_e$", "theta_ef": r"$\theta_{ef}$ (frozen)"}

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import isentropic as iso
from isentropic import thermo

def run_files(run, prefix):
    """3D files of one run, sorted by name (i.e. by time step), and its reference profile file (or None)."""
    files = sorted(run.glob(FILES_3D.format(prefix=prefix)))
    stat = sorted(run.glob(FILE_STAT.format(prefix=prefix))) if FILE_STAT else []
    return files, (stat[0] if stat else None)


for exp, (run, prefix) in RUNS.items():
    files, stat = run_files(run, prefix)
    print(f"{exp}: {len(files)} 3D files, reference profile: {stat}")
    assert files, f"no 3D files in {run}"

files0, stat0 = run_files(*RUNS[next(iter(RUNS))])
ds0 = xr.open_dataset(files0[0])
z = ds0[Z_DIM].values
ds0

## Thermodynamic fields at the first time of the first run

In [ ]:
def reference_density(stat):
    """RHO(time, z) from the reference profile file, or None."""
    if stat is None:
        return None
    with xr.open_dataset(stat) as st:
        if RHO_VAR not in st:
            print(f"{RHO_VAR} not in {stat.name}, computing the density instead")
            return None
        rho = st[RHO_VAR].load()
    assert rho.sizes[Z_DIM] == len(z), "reference profile and 3D output have different vertical levels"
    return rho


def load_time(ds, it, rho_ref=None):
    """Return rho*w, theta_e, frozen theta_e (nz, ny, nx) and the density profile (nz) at time index it.

    The density is the reference profile at the closest time if rho_ref is given, otherwise the
    horizontal mean of the density computed from the snapshot."""
    snap = ds.isel({TIME_DIM: it})
    get = lambda name: snap[name].values.astype(float)
    T = get(VAR["T"])
    qv = np.maximum(get(VAR["qv"]) * 1e-3, 1e-10)           # g/kg -> kg/kg
    qn = np.maximum(get(VAR["qn"]) * 1e-3, 0.0)
    liq_frac = np.clip((T - T_ICE) / (T_LIQ - T_ICE), 0.0, 1.0)
    ql, qi = qn * liq_frac, qn * (1.0 - liq_frac)
    p = get(VAR["p"])[:, None, None] * 100.0 + get(VAR["pp"])   # mb -> Pa, plus perturbation
    coords = {
        "theta_e": thermo.equivalent_potential_temperature(p, T, qv, ql, qi),
        "theta_ef": thermo.frozen_equivalent_potential_temperature(p, T, qv, ql, qi),
    }
    if rho_ref is None:
        rho = thermo.density(p, T, qv, ql, qi).mean((1, 2))    # density profile (anelastic)
    else:
        rho = rho_ref.sel({TIME_DIM: snap[TIME_DIM].values}, method="nearest").values
    return rho[:, None, None] * get(VAR["w"]), coords, rho


rho_ref0 = reference_density(stat0)
rhow, coords, rho = load_time(ds0, 0, rho_ref0)
if rho_ref0 is not None:
    print("max relative difference, reference vs computed density:",
          np.max(np.abs(rho / load_time(ds0, 0)[2] - 1)))
for name, th in coords.items():
    print(f"{name:8s} range: {np.nanmin(th):.1f} - {np.nanmax(th):.1f} K   bins: {BINS[name]}")

## Conditional sums for each experiment and time
The result has one set of variables per coordinate: `MF_theta_e(experiment, time, z, theta_e)`,
`MF_theta_ef(experiment, time, z, theta_ef)`, and the same for `AREA` and `ATH` (sum of the coordinate itself).

In [ ]:
centers = {name: b["th_min"] + (b["th_max"] - b["th_min"]) / (b["nbin"] - 1) * np.arange(b["nbin"])
           for name, b in BINS.items()}
dth = {name: c[1] - c[0] for name, c in centers.items()}


def process_experiment(exp, run, prefix):
    """Conditional sums in both coordinates at every selected time of one run."""
    files, stat = run_files(run, prefix)
    rho_ref = reference_density(stat)
    # (file, time index in file) for every snapshot of the run, then select with TIME_SLICE
    snaps = []
    for f in files:
        with xr.open_dataset(f) as ds:
            snaps += [(f, it) for it in range(ds.sizes[TIME_DIM])]
    snaps = snaps[TIME_SLICE]
    out = {}
    for name, b in BINS.items():
        for v in ("MF", "AREA", "ATH"):
            out[f"{v}_{name}"] = np.zeros((len(snaps), len(z), b["nbin"]))
    RHO = np.zeros((len(snaps), len(z)))
    time = np.zeros(len(snaps))
    for k, (f, it) in enumerate(snaps):
        with xr.open_dataset(f) as ds:
            time[k] = ds[TIME_DIM].values[it]
            rhow, coords, RHO[k] = load_time(ds, it, rho_ref)
        for name, th in coords.items():
            mask = ~np.isfinite(th)
            (mf, ath), area = iso.iso_multi_integral(np.stack([rhow, th]), th, mask, 1.0, **BINS[name])
            out[f"MF_{name}"][k], out[f"ATH_{name}"][k], out[f"AREA_{name}"][k] = mf, ath, area
        if k % 24 == 0:
            print(exp, "snapshot", k, "/", len(snaps), f.name)
    data = {key: (("time", "z", key.split("_", 1)[1]), val) for key, val in out.items()}
    data["RHO"] = (("time", "z"), RHO)
    return xr.Dataset(data, coords={"time": time, "z": z, **centers}).expand_dims(experiment=[exp])


sums = xr.concat([process_experiment(exp, *r) for exp, r in RUNS.items()], dim="experiment", join="outer")
sums

## Streamfunctions
$\Psi$ is computed for every experiment and time in each coordinate, at the upper edge of each bin.

In [ ]:
result = sums.copy()
for name in BINS:
    mf, area = sums[f"MF_{name}"].values, sums[f"AREA_{name}"].values
    ts = iso.two_stream(mf, area)
    result[f"PSI_{name}"] = (("experiment", "time", "z", name), iso.streamfunction(mf, area))
    result[f"PSI_{name}"].attrs.update(long_name=f"isentropic streamfunction in {name}", units="kg m-2 s-1",
                                       note=f"evaluated at {name} + d{name}/2")
    result[f"MEAN_{name}"] = sums[f"ATH_{name}"].sum(name) / sums[f"AREA_{name}"].sum(name)
    result[f"M_UP_{name}"] = (("experiment", "time", "z"), ts["M_up"])
    result[f"M_DOWN_{name}"] = (("experiment", "time", "z"), ts["M_down"])
result.attrs.update(source=", ".join(str(r) for r, _ in RUNS.values()), **{f"d{name}": d for name, d in dth.items()})
result

## Comparison, one figure per experiment
Top: time-mean $\Psi$ in $\theta_e$ and in frozen $\theta_e$, with the same colour scale and the mean
profile of each coordinate. Bottom: the two-stream ascending and descending mass fluxes $M^\pm$, and the
overturning $\max_\theta |\Psi|$ in each coordinate. Both top panels use the same $\theta$ axis.

In [ ]:
# Time means: sum the conditional sums over time, then compute Psi
tm = {}
for name in BINS:
    mf = sums[f"MF_{name}"].sum("time", skipna=True).values
    area = sums[f"AREA_{name}"].sum("time", skipna=True).values
    ts = iso.two_stream(mf, area)
    tm[name] = dict(psi=iso.streamfunction(mf, area),
                    mean=sums[f"ATH_{name}"].sum("time", skipna=True).sum(name).values / area.sum(-1),
                    M_up=ts["M_up"], M_down=ts["M_down"])
lim = max(np.nanmax(np.abs(t["psi"])) for t in tm.values())
levels = np.linspace(-lim, lim, 20)
xlim = (min(b["th_min"] for b in BINS.values()), max(b["th_max"] for b in BINS.values()))

for e, exp in enumerate(sums.experiment.values):
    fig, axs = plt.subplots(2, 2, figsize=(12, 9), sharey=True)
    for ax, name in zip(axs[0], BINS):
        th = centers[name] + dth[name] / 2
        pc = ax.pcolormesh(th, z / 1e3, tm[name]["psi"][e], cmap="RdBu_r", vmin=-lim, vmax=lim, shading="auto")
        ax.contour(th, z / 1e3, tm[name]["psi"][e], levels=levels, colors="k", linewidths=0.4)
        ax.plot(tm[name]["mean"][e], z / 1e3, "k", lw=2, label=f"mean {LABEL[name]}")
        ax.set_xlabel(LABEL[name] + " (K)")
        ax.set_xlim(xlim)
        ax.legend(loc="upper right")
    axs[0, 0].set_ylabel("z (km)")
    fig.colorbar(pc, ax=axs[0], label=r"$\Psi$ (kg m$^{-2}$ s$^{-1}$)")

    ax = axs[1, 0]
    for name, ls in zip(BINS, ["-", "--"]):
        ax.plot(tm[name]["M_up"][e], z / 1e3, "C0", ls=ls, label=r"$M^+$, " + LABEL[name])
        ax.plot(-tm[name]["M_down"][e], z / 1e3, "C1", ls=ls, label=r"$-M^-$, " + LABEL[name])
    ax.set_xlabel(r"mass flux (kg m$^{-2}$ s$^{-1}$)")
    ax.set_ylabel("z (km)")
    ax.legend()

    ax = axs[1, 1]
    for name, ls in zip(BINS, ["-", "--"]):
        ax.plot(np.nanmax(np.abs(tm[name]["psi"][e]), axis=-1), z / 1e3, "k", ls=ls, label=LABEL[name])
    ax.set_xlabel(r"$\max_\theta |\Psi|$ (kg m$^{-2}$ s$^{-1}$)")
    ax.legend()

    nt = int(np.isfinite(sums.MF_theta_e.isel(experiment=e, z=0, theta_e=0)).sum())
    fig.suptitle(f"{exp}: time mean over {nt} snapshots")
    plt.show()

## Save

In [ ]:
result.to_netcdf("iso_streamfunction_sam_rce.nc")